# Vietnamese Legal RAG - R2AI2026
Self-contained Kaggle notebook: data → indexing → retrieval → submission

**Setup:**
- Enable GPU (T4/P100) in Kaggle settings
- Upload `R2AIStage1DATA.json` as a Kaggle dataset
- Run all cells → download notebook with outputs when done

In [ ]:
# 0. Install dependencies
!pip install -q sentence-transformers rank-bm25 pyvi faiss-cpu datasets pyyaml tqdm
print('Dependencies installed.')

In [ ]:
import json, os, re, pickle, time, sys, zipfile, glob
from collections import defaultdict
from tqdm import tqdm
import numpy as np
import yaml

# Fix Windows encoding (no-op on Kaggle Linux)
if sys.platform == 'win32':
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# Config
CFG = {
    'embedding': {'model': 'BAAI/bge-m3', 'max_length': 512, 'batch_size': 32},
    'retrieval': {'dense_top_k': 50, 'bm25_top_k': 50, 'rrf_k': 60},
    'reranker': {'model': 'BAAI/bge-reranker-v2-m3', 'top_n': 10},
}
BM25_TOP_K = 100  # retrieve broad for F2 (Recall weighted)
FINAL_TOP_N = 10

print('Config loaded.')

## Step 1: Data Collection (phapdien + UTS_VLC)

In [ ]:
# 1a. Download phapdien from HuggingFace → articles
from datasets import load_dataset
from collections import defaultdict

ARTICLES_PATH = 'articles.jsonl'

# ── phapdien: law_id from source_note_text, article_title = "Điều X" ──
_LAW_ID_PATTERNS = [
    re.compile(r'(\d{1,3}/\d{4}/QH\d+)'),
    re.compile(r'(\d{1,3}/\d{4}/NĐ-CP)'),
    re.compile(r'(\d{1,3}/\d{4}/ND-CP)'),
    re.compile(r'(\d{1,3}/\d{4}/TT-[A-ZĐ]+)'),
    re.compile(r'(\d{1,3}/\d{4}/TTLT-[A-ZĐ]+)'),
    re.compile(r'(\d{1,3}/\d{4}/QĐ-[A-ZĐ]+)'),
    re.compile(r'(\d{1,3}/\d{4}/QD-[A-ZĐ]+)'),
    re.compile(r'(\d{1,3}/\d{4}/PL-UBTVQH\d+)'),
    re.compile(r'(\d{1,3}/\d{4}/NQ-[A-ZĐ]+)'),
]

def extract_law_id(source_note):
    if not source_note:
        return ''
    for pat in _LAW_ID_PATTERNS:
        m = pat.search(source_note)
        if m:
            return m.group(1)
    return ''

def build_law_name_phapdien(source_note, law_id):
    if not source_note:
        return law_id
    loai_order = ['Bộ luật', 'Luật', 'Pháp lệnh', 'Nghị định', 'Thông tư', 'Quyết định', 'Nghị quyết']
    loai_vb = ''
    for vn_type in loai_order:
        if vn_type in source_note:
            loai_vb = vn_type
            break
    if not loai_vb:
        return law_id
    if law_id:
        idx = source_note.find(law_id)
        if idx != -1:
            remainder = source_note[idx + len(law_id):].strip()
            for stop in ['ngày', 'của Quốc hội', 'của Chính phủ', 'của Bộ', 'của Ngân hàng',
                         'của Thủ tướng', 'của Ủy ban', ', có hiệu lực', '  ']:
                ri = remainder.find(stop)
                if ri > 0:
                    remainder = remainder[:ri].strip()
            trich_yeu = remainder.strip(' ,.')
            if trich_yeu:
                return f'{loai_vb} {law_id} {trich_yeu}'
    return f'{loai_vb} {law_id}'

print('Downloading phapdien from HuggingFace...')
ds = load_dataset('tmquan/phapdien-moj-gov-vn', split='train')
print(f'  Raw rows: {len(ds)}')

phapdien_groups = defaultdict(list)
phapdien_law_names = {}
phapdien_chapters = {}
skipped = 0

for item in tqdm(ds, desc='Processing phapdien'):
    source_note = item.get('source_note_text', '')
    law_id = extract_law_id(source_note)
    if not law_id:
        skipped += 1
        continue
    content = item.get('content_text', '')
    if not content or len(content.strip()) < 10:
        skipped += 1
        continue
    article_title = item.get('article_title', '')
    m = re.match(r'Điều\s+(\d+)', article_title)
    if not m:
        continue
    article_num = f'Điều {m.group(1)}'
    key = (law_id, article_num)
    phapdien_groups[key].append(content)
    if key not in phapdien_law_names:
        phapdien_law_names[key] = build_law_name_phapdien(source_note, law_id)
        phapdien_chapters[key] = item.get('chapter_title', '')

print(f'  Skipped: {skipped}')
print(f'  Unique (law_id, article_num): {len(phapdien_groups)}')

phapdien_articles = []
for (law_id, article_num), texts in phapdien_groups.items():
    seen_texts = list(dict.fromkeys(texts))
    phapdien_articles.append({
        'law_id': law_id,
        'law_name': phapdien_law_names.get((law_id, article_num), law_id),
        'article_num': article_num,
        'chapter': phapdien_chapters.get((law_id, article_num), ''),
        'text': '\n'.join(seen_texts),
        'source': 'phapdien',
    })
print(f'  phapdien articles: {len(phapdien_articles)}')

In [ ]:
# 1b. Download UTS_VLC from HuggingFace → parse markdown into Điều-level chunks
# This is the BIG source (~58K articles vs phapdien's ~4K)

UTS_SPLITS = ["2026", "2026_01", "2023", "2021"]

DIEU_RE = re.compile(r"^\s*Điều\s+(\d+[a-z]?)\s*[\.\:]\s*(.*)", re.IGNORECASE)
CHUONG_RE = re.compile(r"^\s*Chương\s+([IVXLCDM]+|\d+)\s*[\.\-\s]+(.*)", re.IGNORECASE)

# Slug-to-law_id mapping for UTS_VLC doc ids that aren't standard format
_SLUG_MAP = {
    "bo-luat-dan-su": "91/2015/QH13",
    "Bo-luat-dan-su-2015-296215": "91/2015/QH13",
    "code-2015-bo-luat-dan-su": "91/2015/QH13",
    "bo-luat-lao-dong": "45/2019/QH14",
    "Bo-Luat-lao-dong-2019-333670": "45/2019/QH14",
    "code-2019-bo-luat-lao-dong": "45/2019/QH14",
    "luat-dau-tu": "61/2020/QH14",
    "law-2020-luat-dau-tu": "61/2020/QH14",
    "luat-doanh-nghiep": "59/2020/QH14",
    "law-2020-luat-doanh-nghiep": "59/2020/QH14",
    "Luat-Doanh-nghiep-so-59-2020-QH14-427301": "59/2020/QH14",
    "law-2017-luat-ho-tro-doanh-nghiep-nho-va-vua": "04/2017/QH14",
    "Luat-Ho-tro-doanh-nghiep-nho-va-vua-2017-320905": "04/2017/QH14",
    "law-2019-luat-quan-ly-thue": "38/2019/QH14",
    "Luat-quan-ly-thue-2019-387595": "38/2019/QH14",
    "Luat-Bao-hiem-xa-hoi-2014-259700": "71/2015/QH13",
    "law-2014-luat-bao-hiem-xa-hoi": "71/2015/QH13",
    "luat-bao-hiem-xa-hoi": "71/2015/QH13",
    "Luat-Thuong-mai-2005-36-2005-QH11-2633": "36/2005/QH11",
    "law-2005-luat-thuong-mai": "36/2005/QH11",
    "luat-thuong-mai": "36/2005/QH11",
    "Luat-canh-tranh-345182": "23/2018/QH14",
    "law-2018-luat-canh-tranh": "23/2018/QH14",
    "luat-canh-tranh": "23/2018/QH14",
}

# Vietnamese title → law_id for building law_name
_TITLE_TO_ID = {
    "Bộ Luật dân sự": "91/2015/QH13",
    "Bộ luật Lao động": "45/2019/QH14",
    "Luật Đầu tư": "61/2020/QH14",
    "Luật Doanh nghiệp": "59/2020/QH14",
    "Luật Hỗ trợ doanh nghiệp nhỏ và vừa": "04/2017/QH14",
    "Luật Quản lý thuế": "38/2019/QH14",
}

def normalize_uts_law_id(raw_id):
    if re.match(r"^\d{2}/\d{4}/", raw_id):
        return raw_id
    m = re.search(r"(\d{1,3})-(\d{4})-(QH\d+)", raw_id)
    if m:
        return f"{m.group(1)}/{m.group(2)}/{m.group(3)}"
    return _SLUG_MAP.get(raw_id, raw_id)

def build_law_name_uts(doc, law_id):
    doc_type = doc.get("type", "")
    title = doc.get("title", "")
    type_map = {"code": "Bộ luật", "law": "Luật", "ordinance": "Pháp lệnh",
                "decree": "Nghị định", "circular": "Thông tư", "decision": "Quyết định", "resolution": "Nghị quyết"}
    loai_vb = type_map.get(doc_type, "Luật")
    trich_yeu = ""
    for t, lid in _TITLE_TO_ID.items():
        if lid == law_id and any(ord(c) > 127 for c in t):
            trich_yeu = t
            break
    if not trich_yeu and any(ord(c) > 127 for c in title):
        trich_yeu = title
    return f"{loai_vb} {law_id} {trich_yeu}".strip() if trich_yeu else f"{loai_vb} {law_id}"

def parse_uts_markdown(content, law_id, law_name):
    lines = content.split("\n")
    articles = []
    current_chapter = ""
    current_dieu_num = None
    current_dieu_lines = []

    def flush():
        if current_dieu_num is None:
            return
        text = "\n".join(current_dieu_lines).strip()
        if text:
            articles.append({
                "law_id": law_id,
                "law_name": law_name,
                "article_num": f"Điều {current_dieu_num}",
                "chapter": current_chapter,
                "text": text,
                "source": "UTS_VLC",
            })

    for line in lines:
        chuong_match = CHUONG_RE.match(line)
        if chuong_match:
            current_chapter = line.strip()
            continue
        dieu_match = DIEU_RE.match(line)
        if dieu_match:
            flush()
            current_dieu_num = dieu_match.group(1)
            current_dieu_lines = [f"Điều {current_dieu_num}. {dieu_match.group(2).strip()}"]
            continue
        if current_dieu_num is not None:
            current_dieu_lines.append(line)
    flush()
    return articles

uts_articles = []
for split in UTS_SPLITS:
    print(f'Loading UTS_VLC split={split}...')
    try:
        ds = load_dataset('undertheseanlp/UTS_VLC', split=split)
    except Exception as e:
        print(f'  Split {split} failed: {e}. Skipping.')
        continue
    print(f'  {len(ds)} docs in {split}')
    for doc in tqdm(ds, desc=f'Parsing {split}'):
        law_id = normalize_uts_law_id(doc.get("id", ""))
        law_name = build_law_name_uts(doc, law_id)
        content = doc.get("content", "")
        if not content:
            continue
        parsed = parse_uts_markdown(content, law_id, law_name)
        uts_articles.extend(parsed)
    print(f'  Running total: {len(uts_articles)} articles')

print(f'\nUTS_VLC total: {len(uts_articles)} articles')

## Step 2: Load Articles

In [ ]:
# 2. Merge UTS_VLC + phapdien, deduplicate, save to articles.jsonl
all_articles = uts_articles + phapdien_articles
print(f'Before dedup: {len(all_articles)} articles (UTS_VLC: {len(uts_articles)}, phapdien: {len(phapdien_articles)})')

# Deduplicate by (law_id, article_num) — prefer UTS_VLC over phapdien
seen = {}
for art in all_articles:
    key = (art['law_id'], art['article_num'])
    if key not in seen:
        seen[key] = art
    else:
        existing = seen[key]
        if existing['source'] != 'UTS_VLC' and art['source'] == 'UTS_VLC':
            seen[key] = art

articles = list(seen.values())
print(f'After dedup: {len(articles)} articles')

# Save to disk
with open(ARTICLES_PATH, 'w', encoding='utf-8') as f:
    for art in articles:
        f.write(json.dumps(art, ensure_ascii=False) + '\n')
print(f'Saved to {ARTICLES_PATH}')

law_counts = defaultdict(int)
for art in articles:
    law_counts[art['law_id']] += 1
print(f'Unique laws: {len(law_counts)}')
# Show top laws by article count
for law_id, count in sorted(law_counts.items(), key=lambda x: -x[1])[:10]:
    print(f'  {law_id}: {count} articles')

## Step 3: BM25 Index

In [ ]:
# 3. Build BM25 index with pyvi tokenization
from pyvi.ViTokenizer import tokenize as vi_tokenize
from rank_bm25 import BM25Okapi

def tokenize_vi(text):
    try:
        return vi_tokenize(text).split()
    except Exception:
        return text.split()

BM25_PATH = 'bm25_index.pkl'

if os.path.exists(BM25_PATH):
    print('Loading existing BM25 index...')
    with open(BM25_PATH, 'rb') as f:
        bm25_data = pickle.load(f)
    bm25 = bm25_data['index']
    print(f'  BM25 corpus_size: {bm25.corpus_size}')
else:
    print('Building BM25 index...')
    tokenized_corpus = []
    for art in tqdm(articles, desc='Tokenizing'):
        parts = [art['article_num']]
        if art.get('law_name'): parts.append(art['law_name'])
        if art.get('chapter'): parts.append(art['chapter'])
        parts.append(art['text'])
        tokens = tokenize_vi('. '.join(parts))
        tokenized_corpus.append(tokens)

    bm25 = BM25Okapi(tokenized_corpus)
    print(f'  BM25 corpus_size: {bm25.corpus_size}')

    with open(BM25_PATH, 'wb') as f:
        pickle.dump({'index': bm25}, f)
    print(f'  Saved to {BM25_PATH}')

## Step 4: Dense Embeddings (BGE-M3)

In [ ]:
# 4. Encode articles with BGE-M3
import torch
import faiss
from sentence_transformers import SentenceTransformer

DENSE_PATH = 'dense.npy'
FAISS_PATH = 'index.faiss'
META_PATH = 'metadata.jsonl'

def build_texts(articles):
    texts = []
    for art in articles:
        parts = [art['article_num']]
        if art.get('law_name'): parts.append(art['law_name'])
        if art.get('chapter'): parts.append(art['chapter'])
        parts.append(art['text'])
        texts.append('. '.join(parts))
    return texts

# Check if FAISS index already exists
if os.path.exists(FAISS_PATH):
    print('FAISS index already exists. Loading...')
    faiss_index = faiss.read_index(FAISS_PATH)
    print(f'  Index size: {faiss_index.ntotal} vectors')
elif os.path.exists(DENSE_PATH):
    print('Dense vectors exist. Building FAISS...')
    dense_vecs = np.load(DENSE_PATH).astype('float32')
    faiss.normalize_L2(dense_vecs)
    faiss_index = faiss.IndexFlatIP(dense_vecs.shape[1])
    faiss_index.add(dense_vecs)
    faiss.write_index(faiss_index, FAISS_PATH)
    print(f'  FAISS index: {faiss_index.ntotal} vectors, dim={faiss_index.d}')
    del dense_vecs  # free memory
else:
    # Auto-detect GPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cuda':
        gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
        print(f'GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f}GB)')
    print(f'Encoding on {device}...')

    model_name = CFG['embedding']['model']
    batch_size = CFG['embedding']['batch_size']
    max_length = CFG['embedding']['max_length']

    model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    texts = build_texts(articles)
    print(f'Encoding {len(texts)} texts (batch_size={batch_size}, max_length={max_length})...')

    # Encode with checkpointing
    checkpoint_every = 5000
    all_embeddings = []
    start_idx = 0

    # Resume from checkpoint if exists
    partial_path = 'dense_partial.npy'
    if os.path.exists(partial_path):
        partial = np.load(partial_path)
        if partial.shape[0] < len(texts):
            start_idx = partial.shape[0]
            all_embeddings = [partial]
            print(f'  Resuming from {start_idx}/{len(texts)}')

    for i in tqdm(range(start_idx, len(texts), batch_size), desc='Encoding'):
        batch = texts[i:i + batch_size]
        emb = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        all_embeddings.append(np.array(emb, dtype='float32'))

        # Checkpoint
        done = min(i + batch_size, len(texts))
        if done % checkpoint_every < batch_size:
            combined = np.vstack(all_embeddings)
            np.save(partial_path, combined)
            print(f'  Checkpoint: {combined.shape[0]}/{len(texts)}')

    dense_vecs = np.vstack(all_embeddings)
    np.save(DENSE_PATH, dense_vecs)
    print(f'Dense shape: {dense_vecs.shape}')

    # Build FAISS
    faiss.normalize_L2(dense_vecs)
    faiss_index = faiss.IndexFlatIP(dense_vecs.shape[1])
    faiss_index.add(dense_vecs)
    faiss.write_index(faiss_index, FAISS_PATH)
    print(f'FAISS index: {faiss_index.ntotal} vectors')

    del dense_vecs, model  # free GPU memory for reranker
    if device == 'cuda':
        torch.cuda.empty_cache()

# Save metadata
if not os.path.exists(META_PATH):
    with open(META_PATH, 'w', encoding='utf-8') as f:
        for art in articles:
            f.write(json.dumps({
                'law_id': art['law_id'],
                'law_name': art['law_name'],
                'article_num': art['article_num'],
                'chapter': art.get('chapter', ''),
            }, ensure_ascii=False) + '\n')
    print(f'Saved metadata to {META_PATH}')

# Load metadata for retrieval
metadata = []
with open(META_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        metadata.append(json.loads(line))
print(f'Loaded {len(metadata)} metadata entries')

## Step 5: Reranker

In [ ]:
# 5. Load cross-encoder reranker (GPU)
use_reranker = False
if torch.cuda.is_available():
    print('Loading BGE-reranker-v2-m3 on GPU...')
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    reranker_model_name = CFG['reranker']['model']
    reranker_tokenizer = AutoTokenizer.from_pretrained(reranker_model_name)
    reranker_model = AutoModelForSequenceClassification.from_pretrained(reranker_model_name)
    reranker_model.to('cuda')
    reranker_model.eval()
    use_reranker = True
    print('Reranker loaded.')
else:
    print('No GPU, skipping reranker.')

def rerank(query, candidates, top_n=10):
    pairs = [[query, art['text']] for art in candidates]
    with torch.no_grad():
        features = reranker_tokenizer(pairs, padding=True, truncation=True, max_length=512, return_tensors='pt')
        features = {k: v.to('cuda') for k, v in features.items()}
        scores = reranker_model(**features).logits.squeeze(-1).float().tolist()
    if isinstance(scores, float):
        scores = [scores]
    for art, score in zip(candidates, scores):
        art['rerank_score'] = float(score)
    return sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)[:top_n]

## Step 6: Retrieval

In [ ]:
# 6. Load competition questions
search_paths = [
    'R2AIStage1DATA.json',
    '/kaggle/input/r2ai-stage1/R2AIStage1DATA.json',
    '/kaggle/input/r2ai2026/R2AIStage1DATA.json',
]
input_path = None
for p in search_paths:
    if os.path.exists(p):
        input_path = p
        break
if not input_path:
    matches = glob.glob('/kaggle/input/**/R2AIStage1DATA.json', recursive=True)
    if matches:
        input_path = matches[0]

if not input_path:
    print('ERROR: R2AIStage1DATA.json not found!')
    print('Upload it as a Kaggle dataset.')
else:
    with open(input_path, 'r', encoding='utf-8') as f:
        questions = json.load(f)
    print(f'Loaded {len(questions)} questions from {input_path}')

In [ ]:
# 7. Load query encoder for dense search
device = 'cuda' if torch.cuda.is_available() else 'cpu'
query_model = SentenceTransformer(CFG['embedding']['model'], trust_remote_code=True, device=device)
print(f'Query encoder loaded on {device}')

In [ ]:
# 8. Run hybrid retrieval + reranking on all questions

def dense_search(query_text, top_k=50):
    q_emb = query_model.encode([query_text], normalize_embeddings=True)
    q_vec = q_emb.astype('float32')
    faiss.normalize_L2(q_vec)
    scores, ids = faiss_index.search(q_vec, top_k)
    results = []
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0])):
        if idx < 0: continue
        results.append({'index': int(idx), 'score': float(score), 'rank': rank})
    return results

def bm25_search(query_text, top_k=100):
    tokens = tokenize_vi(query_text)
    scores = bm25.get_scores(tokens)
    top_indices = scores.argsort()[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_indices):
        if scores[idx] <= 0: continue
        results.append({'index': int(idx), 'score': float(scores[idx]), 'rank': rank})
    return results

def rrf_merge(dense_results, bm25_results, k=60):
    scores = {}
    for r in dense_results:
        idx = r['index']
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + r['rank'] + 1)
    for r in bm25_results:
        idx = r['index']
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + r['rank'] + 1)
    merged = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [{'index': idx, 'rrf_score': score} for idx, score in merged]

results = []
total = len(questions)
start_time = time.time()

for i, item in enumerate(questions):
    qid = item['id']
    question = item['question']

    # Hybrid retrieval
    dense_res = dense_search(question, top_k=CFG['retrieval']['dense_top_k'])
    bm25_res = bm25_search(question, top_k=BM25_TOP_K)
    merged = rrf_merge(dense_res, bm25_res, k=CFG['retrieval']['rrf_k'])

    # Top candidates with metadata
    candidates = []
    for r in merged[:30]:
        idx = r['index']
        meta = metadata[idx]
        candidates.append({
            'law_id': meta['law_id'],
            'law_name': meta['law_name'],
            'article_num': meta['article_num'],
            'chapter': meta.get('chapter', ''),
            'text': articles[idx]['text'],
            'rrf_score': r['rrf_score'],
        })

    # Rerank
    if use_reranker and candidates:
        try:
            candidates = rerank(question, candidates[:20], top_n=FINAL_TOP_N)
        except Exception:
            candidates = candidates[:FINAL_TOP_N]
    else:
        candidates = candidates[:FINAL_TOP_N]

    # Build output
    seen_docs, seen_arts = set(), set()
    relevant_docs, relevant_articles = [], []
    for art in candidates:
        doc_key = f"{art['law_id']}|{art['law_name']}"
        art_key = f"{art['law_id']}|{art['law_name']}|{art['article_num']}"
        if doc_key not in seen_docs:
            seen_docs.add(doc_key)
            relevant_docs.append(doc_key)
        if art_key not in seen_arts:
            seen_arts.add(art_key)
            relevant_articles.append(art_key)

    results.append({
        'id': qid,
        'question': question,
        'answer': '',
        'relevant_docs': relevant_docs,
        'relevant_articles': relevant_articles,
    })

    if (i + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        eta = (total - i - 1) / rate if rate > 0 else 0
        print(f'  [{i+1}/{total}] rate={rate:.1f} q/s, ETA={eta/60:.1f} min')

    if (i + 1) % 200 == 0:
        with open('results_checkpoint.json', 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

total_time = time.time() - start_time
print(f'\nDone! {total} questions in {total_time:.1f}s ({total_time/60:.1f} min)')

## Step 7: Submit

In [ ]:
# 9. Save results and create submission zip
with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f'Saved results.json ({len(results)} entries)')

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('results.json')
print('Created submission.zip')

# Stats
doc_counts = [len(r['relevant_docs']) for r in results]
art_counts = [len(r['relevant_articles']) for r in results]
print(f'\nAvg relevant_docs: {sum(doc_counts)/len(doc_counts):.1f}')
print(f'Avg relevant_articles: {sum(art_counts)/len(art_counts):.1f}')

# Validate
errors = []
for i, r in enumerate(results):
    for field in ['id', 'question', 'answer', 'relevant_docs', 'relevant_articles']:
        if field not in r:
            errors.append(f'Entry {i}: missing {field}')
    for doc in r.get('relevant_docs', []):
        if '|' not in doc:
            errors.append(f'Entry {i}: bad doc format')
    for art in r.get('relevant_articles', []):
        if len(art.split('|')) < 3:
            errors.append(f'Entry {i}: bad article format')
if errors:
    print(f'\nValidation errors: {len(errors)}')
    for e in errors[:5]: print(f'  - {e}')
else:
    print('\nValidation: ALL PASSED')